# Data Restructuring

References:

[1] McKinney, Wes. *Python for data analysis.* " O'Reilly Media, Inc.", 2022.

[2] VanderPlas, Jake. *Python data science handbook: Essential tools for working with data. Second Edition* " O'Reilly Media, Inc.", 2023.

[3] Johansson, Robert, Robert Johansson, and Suresh John. *Numerical python.* Vol. 1. New York: Apress, 2019.

Data, in many application, may be spread across several files or database, or be arranged in a form that is not convenient to analyze. Thus, data restructuring is crucial for enhancing the organization, quality, and utility of data.

Well-structured data facilitates compatibility, integration, and efficient storage enabling accurate modeling and machine learning.

In [ ]:
import numpy as np
import pandas as pd

## 1 Combining and Merging Datasets

In [ ]:
transform = np.array([
                [0, -1, 0],
                [-1, 0, 0],
                [0, 0, 1],
            )

Data contained in pandas objects can be combined in a number of ways:

- `pandas.merge` – connect rows in DataFrames based on one or more keys. This is similar to SQL’s join operations
- `pandas.concat` – concatenate or “stack” objects along an axis. Similar to numpy’s concatenate.
- `combine_first` – splice together overlapping data to fill in missing values in one object with values from another.

### Database-Style Data Frame Joins

In [ ]:
df1 = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "a", "b"],
                    "data1": pd.Series(range(7), dtype="Int64")})
df2 = pd.DataFrame({"key": ["a", "b", "d"],
                    "data2": pd.Series(range(3), dtype="Int64")})

In [ ]:
df1

In [ ]:
df2

Calling `pd.merge` will perofrm a many-to-one join

In [ ]:
pd.merge(df1, df2)

Note that we didn't specify a column to join on. If that information is not specified the overlapping column names will be used as keys. It is often good practice to explicitly specify it though.

In [ ]:
pd.merge(df1, df2, on='key')

By default, pandas does an `inner` join, the resulting keys are the intersection or the common set found in both tables. Other possible options are `left`, `right`, `outer`, and `cross`.

In [ ]:
pd.merge(df1, df2, how='outer')

In [ ]:
pd.merge(df1, df2, how='left')

In [ ]:
pd.merge(df1, df2, how='right')

In [ ]:
pd.merge(df1, df2, how='cross')

A basic kind of time series object in pandas is a `Series` indexed by timestamps.

In [ ]:
rng = np.random.default_rng(1337)
time_series = pd.Series(rng.integers(10, size=(1_000)),
                        index=pd.date_range('2022-01-01', periods=1_000))
time_series

Under the hood, `datetime` objects have been put in the `DatetimeIndex`.

In [ ]:
time_series.index

### Indexing, Selection, Subsetting

Time series behaves like any other `Series` that we have indexed

In [ ]:
time_series['2024-03-09']

In [ ]:
time_series['2024-03-09':'2024-06-12']

Slicing with `datetime` objects also works

In [ ]:
time_series[datetime(2024, 3, 9):datetime(2024, 6, 12)]

In [ ]:
start_date = datetime(2024, 3, 9)
time_series[start_date:start_date + timedelta(days=100)]

You can also instead specify which keys are to be join on the left and right dataframes.

In [ ]:
df3 = pd.DataFrame({"lkey": ["b", "b", "a", "c", "a", "a", "b"],
                    "data1": pd.Series(range(7), dtype="Int64")})
df4 = pd.DataFrame({"rkey": ["a", "b", "d"],
                    "data2": pd.Series(range(3), dtype="Int64")})

In [ ]:
df3

In [ ]:
df4

In [ ]:
pd.merge(df3, df4)

In [ ]:
pd.merge(df3, df4, left_on='lkey', right_on='rkey', how='outer')

### Merging on Index

In some cases, the merge key(s) in a DataFrame will be found in the index. In this case we specify the `left_index` or `right_index` to indicate that the index should be used as merge key.

In [ ]:
left1 = pd.DataFrame({"key": ["a", "b", "a", "a", "b", "c"],
                      "value": pd.Series(range(6), dtype="Int64")})
right1 = pd.DataFrame({"group_val": [3.5, 7]}, index=["a", "b"])

In [ ]:
left1

In [ ]:
right1

In [ ]:
pd.merge(left1, right1, left_on='key', right_index=True)

Using indexes of both sides of the merge is also possible.

In [ ]:
left2 = pd.DataFrame([[1., 2.], [3., 4.], [5., 6.]],
                     index=["a", "c", "e"],
                     columns=["Manila", "Pasig"]).astype("Int64")
right2 = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [13, 14]],
                      index=["b", "c", "d", "e"],
                      columns=["Makati", "Pasay"]).astype("Int64")

In [ ]:
left2

In [ ]:
right2

In [ ]:
pd.merge(left2, right2, how='outer', left_index=True, right_index=True)

`DataFrame` has a `join` instance method to simplify merging by index.

In [ ]:
left2.join(right2, how='outer')

`join` also supports a left join on the join keys by default. It supports joining the index of the passed DataFrame on the columns of the calling dataframe.

In [ ]:
left1

In [ ]:
right1

In [ ]:
left1.join(right1, on='key')

In [ ]:
left1.join(right1, on='key', how='inner')

Lastly, for simple index-on-index merges, you can pass a list of DataFrames to `join` as an alternative to the more general `pandas.concat`.

In [ ]:
another = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [16., 17.]],
                       index=["a", "c", "e", "f"],
                       columns=["Quezon City", "Muntinlupa"])
another

In [ ]:
left2

In [ ]:
right2

In [ ]:
left2.join([right2, another])

In [ ]:
left2.join([right2, another], how='outer')

### Concatenating Along an Axis

Another kind of data combination operation is referred as `concatenation` or `stacking`. This is similar to `np.concatenate` but since pandas objects have labeled axes and columns, it allows you to further generalize array concatenation. In particular, you have a number of additional concerns:

- If objects are indexed differently on other axes, should we combine the distinct elements in these axes or use only the values in common?
- Do the concatenated chunks of data need to be identifiable as such in the resulting object?
- Does the "concatenation axis" contain data that needs to be preserved?

In [ ]:
s1 = pd.Series([0, 1], index=["a", "b"], dtype="Int64")
s2 = pd.Series([2, 3, 4], index=["c", "d", "e"], dtype="Int64")
s3 = pd.Series([5, 6], index=["f", "g"], dtype="Int64")

In [ ]:
s1

In [ ]:
s2

In [ ]:
s3

In [ ]:
pd.concat([s1, s2, s3])

By default concat works along the `axis='index'`, producing a nother `Series`. If you pass `axis='columns'`, the result will be a `DataFrame`:

In [ ]:
pd.concat([s1, s2, s3], axis='columns')

In [ ]:
s4 = pd.concat([s1, s3])
s4

In [ ]:
s1

In [ ]:
pd.concat([s1, s4], axis='columns')

In [ ]:
pd.concat([s1, s4], axis='columns', join='inner')

A potential issue is that the concatenated pieces are not identifieable in the result. Suppose instead you wanted to create an hieararchical index on the concatenation axis, we can use the `keys` argument.

In [ ]:
result = pd.concat([s1, s1, s3], keys=['one', 'two', 'three'])
result

We can rotate the inner index as a column by using `unstack()`.

In [ ]:
result.unstack()

In the case of combining `Series` along the columns, the keys become the `DataFrame` column headers:

In [ ]:
pd.concat([s1, s2, s3], axis='columns', keys=['one', 'two', 'three'])

The same logic extends to DataFrame objects:

In [ ]:
df1 = pd.DataFrame(np.arange(6).reshape(3, 2), index=["a", "b", "c"],
                   columns=["one", "two"])
df2 = pd.DataFrame(5 + np.arange(4).reshape(2, 2), index=["a", "c"],
                   columns=["three", "four"])

In [ ]:
df1

In [ ]:
df2

In [ ]:
pd.concat([df1, df2])

In [ ]:
pd.concat([df1, df2], axis='columns')

In [ ]:
pd.concat([df1, df2], keys=['df1', 'df2'])

In [ ]:
pd.concat([df1, df2], axis='columns', keys=['level1', 'level2'])

You can pass instead a dictionary of objects instead of a list, the dictionary's keys will be used for the keys option:

In [ ]:
pd.concat({'level1': df1, 'level2': df2}, axis='columns')

A last consideration concerns when the resutling `DataFrame` contains row index that is irrelevant.

In [ ]:
rng = np.random.default_rng(1337)
df1 = pd.DataFrame(rng.integers(10, size=(3, 4)),
                   columns=["a", "b", "c", "d"]).astype('Int64')
df2 = pd.DataFrame(rng.integers(10, size=(2, 3)),
                   columns=["b", "d", "a"]).astype('Int64')

In [ ]:
df1

In [ ]:
df2

In [ ]:
pd.concat([df1, df2])

In [ ]:
pd.concat([df1, df2], ignore_index=True)

### Combining Data with Overlap

Another scenario in which is different from merging or concatenation is when two datsets have similar indexes that overlap in full or in part. Then you might want to combine or patch the missing values from one dataframe using the other.

In [ ]:
a = pd.Series([np.nan, 2.5, 0.0, 3.5, 4.5, np.nan],
              index=["f", "e", "d", "c", "b", "a"])
b = pd.Series([0., np.nan, 2., np.nan, np.nan, 5.],
              index=["a", "b", "c", "d", "e", "f"])

In [ ]:
a

In [ ]:
b

Here we can use `combine_first` to line up values by index then fill the null values on one series using values of another.

In [ ]:
a.combine_first(b)

In [ ]:
a.fillna(b)

With dataframes `combine_first` does the same thing column by column. So we can think of this as "patching" missing data in the calling object with the data from the object you pass:

In [ ]:
df1 = pd.DataFrame({"a": [1., np.nan, 5., np.nan],
                    "b": [np.nan, 2., np.nan, 6.],
                    "c": range(2, 18, 4)})
df2 = pd.DataFrame({"a": [5., 4., np.nan, 3., 7.],
                    "b": [np.nan, 3., 4., 6., 8.]})

In [ ]:
df1

In [ ]:
df2

In [ ]:
df1.combine_first(df2)

In [ ]:
df1.fillna(df2)

## 2 Reshaping and Pivoting

There are a number of basic operations for rearranging tabular data. These are reshape or pivot operations.
- `stack` – This “rotates” or pivots from the columns in the data to rows
- `unstack` – This pivots from rows into columns.
- `pivot` – convert a data from long format to a wide format.
- `melt` – pivots a data from a wide format to a long format.

### Reshaping with Hierarchical Index

In [ ]:
data = pd.DataFrame(np.arange(6).reshape((2, 3)),
                    index=pd.Index(["Makati", "Pasig"], name="city"),
                    columns=pd.Index(["one", "two", "three"],
                                     name="number"))
data

`stack` pivots the columns into the rows producing a `Series` with an hierarchical index.

In [ ]:
result = data.stack()
result

From a hierarchically indexed `Series`, you can rearrange it back to a `DataFrame` with `unstack`. 

In [ ]:
result.unstack()

By default, the innermost level is unstacked and is used as columns. You can unstack a different level by passing the level number or name.

In [ ]:
result.unstack(level=0)

In [ ]:
result.unstack(level='city')

Unstacking might produce a missing data if all the values in the level aren't found in each subgroup.

In [ ]:
s1 = pd.Series([0, 1, 2, 3], index=["a", "b", "c", "d"], dtype="Int64")
s2 = pd.Series([4, 5, 6], index=["c", "d", "e"], dtype="Int64")
data2 = pd.concat([s1, s2], keys=["one", "two"])
data2

In [ ]:
data2.unstack()

In [ ]:
data2.unstack().stack()

In [ ]:
data2.unstack().stack(future_stack=True)

### Pivoting "Long" to "Wide" Format

A common way to store multiple time series in databases and CSV files is what we called a `long` or stacked format. In this format, individual values are represented by a single row in a table rather than multiple values in a row.

In [ ]:
data = pd.read_csv('data/macrodata.csv',
                   usecols=['year', 'quarter', 'realgdp', 'infl', 'unemp'])
data

Let's create a long format table using this data, first, we consolidate the date time index.

In [ ]:
periods = pd.PeriodIndex.from_fields(
    year=data.pop('year'),
    quarter=data.pop('quarter'))
periods.name = 'date'
periods

In [ ]:
data.index = periods.to_timestamp('D')
data

In [ ]:
data.columns.name = 'item'

In [ ]:
data

We then create the long data by reshaping with `stack`, turn the new index levels into columns with reset_index`.

In [ ]:
long_data = data.stack().to_frame('value').reset_index()
long_data

Data is usually stored this way as we have a fixed schema.

To revert back to the wide format, we use `DataFrame`'s `pivot` method.

In [ ]:
pivoted = long_data.pivot(index='date', columns='item', values='value')
pivoted

If we have a two values that we wanted to reshape simultaneously

In [ ]:
np.random.seed(1337)
long_data["value2"] = np.random.standard_normal(len(long_data))

In [ ]:
long_data

In [ ]:
pivoted = long_data.pivot(index='date', columns='item')
pivoted

`pivot_table` also allows for aggregation between values.

In [ ]:
df = pd.DataFrame({"A": ["foo", "foo", "foo", "foo", "foo",
                         "bar", "bar", "bar", "bar"],
                   "B": ["one", "one", "one", "two", "two",
                         "one", "one", "two", "two"],
                   "C": ["small", "large", "large", "small",
                         "small", "large", "small", "small",
                         "large"],
                   "D": [1, 2, 2, 3, 3, 4, 5, 6, 7],
                   "E": [2, 4, 5, 5, 6, 6, 8, 9, 9]})

In [ ]:
df

In [ ]:
table = pd.pivot_table(df, values='D', index=['A', 'B'],
                       columns=['C'], aggfunc="sum")
table

### Pivoting from "Wide" to "Long" format

The inverse operation for `pivot` is `melt`. Rather than transforming one column to many in a dataframe, we merge multiple columns into one.

In [ ]:
df = pd.DataFrame({"key": ["foo", "bar", "baz"],
                   "A": [1, 2, 3],
                   "B": [4, 5, 6],
                   "C": [7, 8, 9]})
df

In [ ]:
melted = pd.melt(df, id_vars='key')
melted

In [ ]:
pd.melt(df, id_vars="key", value_vars=["A", "B"])

In [ ]:
pd.melt(df, value_vars=["A", "B", "C"])